# CovariateExamples

This notebook is a Python-native tutorial derived from the MATLAB workflow name, implemented from scratch for `nSTAT-python`.

- Execution group: `smoke`
- Workflow family: `signal`
- Paper DOI: `10.1016/j.jneumeth.2012.08.009`
- PMID: `22981419`
- Help page: `docs/help/examples/CovariateExamples.md`


Notebook source link: [CovariateExamples.ipynb](https://github.com/cajigaslab/nSTAT-python/blob/main/notebooks/CovariateExamples.ipynb)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from nstat.analysis import Analysis
from nstat.cif import CIFModel
from nstat.decoding import DecodingAlgorithms
from nstat.events import Events
from nstat.history import HistoryBasis
from nstat.signal import Covariate
from nstat.spikes import SpikeTrain, SpikeTrainCollection
from nstat.trial import CovariateCollection, Trial, TrialConfig

TOPIC = "CovariateExamples"
FAMILY = "signal"
rng = np.random.default_rng(2026)
print(f"Running notebook topic: {TOPIC} (family={FAMILY})")

def validate_numeric_checkpoints(metrics: dict[str, float], limits: dict[str, tuple[float, float]], topic: str) -> None:
    if not metrics:
        raise AssertionError(f"CovariateExamples: CHECKPOINT_METRICS is empty")
    for key, value in metrics.items():
        if not np.isfinite(value):
            raise AssertionError(f"CovariateExamples: metric '{key}' is not finite: {value}")
    for key, (lo, hi) in limits.items():
        if key not in metrics:
            raise AssertionError(f"CovariateExamples: missing checkpoint metric '{key}'")
        value = float(metrics[key])
        if value < float(lo) or value > float(hi):
            raise AssertionError(
                f"CovariateExamples: metric '{key}'={value:.6f} outside [{float(lo):.6f}, {float(hi):.6f}]"
            )
    print(f"Numeric checkpoints for {topic}:", metrics)


In [ ]:
# CovariateExamples: build and inspect multiple covariate signals.
t = np.arange(0.0, 5.0 + 0.01, 0.01)
x = np.exp(-t)
y = np.sin(2.0 * np.pi * t)
z = (-y) ** 3
fx = np.abs(y)
fy = np.abs(y) ** 2

force = Covariate(
    time=t,
    data=np.column_stack([fx, fy]),
    name="Force",
    labels=["f_x", "f_y"],
)
position = Covariate(
    time=t,
    data=np.column_stack([x, y, z]),
    name="Position",
    labels=["x", "y", "z"],
)

# MATLAB figure 1 style: Position covariates with custom line colors.
fig1 = plt.figure(figsize=(9, 5.4))
ax = fig1.add_subplot(1, 1, 1)
ax.plot(t, position.data[:, 0], "g", linewidth=0.5, label="x")
ax.plot(t, position.data[:, 1], "k", linewidth=0.5, label="y")
ax.plot(t, position.data[:, 2], "b", linewidth=0.5, label="z")
ax.set_title(f"{TOPIC}: position covariates")
ax.set_xlabel("time [s]")
ax.legend(loc="upper right")
plt.tight_layout()
plt.show()

# MATLAB figure 2 style: Force original and zero-mean representations.
force_zero_mean = force.data - np.mean(force.data, axis=0, keepdims=True)
fig2, axes = plt.subplots(1, 2, figsize=(10, 4.6), sharex=True)
axes[0].plot(t, force.data[:, 0], "b", linewidth=1.0, label="f_x")
axes[0].plot(t, force.data[:, 1], "k", linewidth=1.0, label="f_y")
axes[0].set_title("Force (original)")
axes[0].set_xlabel("time [s]")
axes[0].legend(loc="upper right")

axes[1].plot(t, force_zero_mean[:, 0], "b", linewidth=1.0, label="f_x")
axes[1].plot(t, force_zero_mean[:, 1], "k", linewidth=1.0, label="f_y")
axes[1].set_title("Force (zero-mean)")
axes[1].set_xlabel("time [s]")
axes[1].legend(loc="upper right")

plt.tight_layout()
plt.show()

assert position.data.shape == (t.size, 3)
assert force.data.shape == (t.size, 2)

CHECKPOINT_METRICS = {
    "position_var": float(np.var(position.data[:, 1])),
    "force_mean": float(np.mean(force.data[:, 0])),
}
CHECKPOINT_LIMITS = {
    "position_var": (0.05, 2.0),
    "force_mean": (0.0, 2.0),
}


In [ ]:
# Execution checkpoints used by CI.
assert TOPIC != "", "Missing topic metadata"
validate_numeric_checkpoints(CHECKPOINT_METRICS, CHECKPOINT_LIMITS, TOPIC)
print("Topic-specific checkpoint for", TOPIC)
print("Notebook checkpoints passed for", TOPIC)


## Next steps

- Compare this notebook with the corresponding MATLAB helpfile output in the validation PDF.
- Use `tools/reports/generate_validation_pdf.py` to run side-by-side parity scoring.
- Refine model assumptions for this specific example until parity thresholds pass.
